# SuperDialseg 지도학습 RoBERTa — 충실 재현 (Colab, standalone)

`github.com/Coldog2333/SuperDialseg` (EMNLP 2023) 의 **supervised RoBERTa** 분절기를
논문 Appendix A 와 repo 코드(`data_collator.py`, `supervised.py`)대로 재현합니다.

### 솔직한 재현 범위 고지

- repo 에는 **학습 스크립트가 없습니다.** `examples/reproduce/main.py` 는 평가 전용이고
  supervised 모델은 거기 연결돼 있지도 않습니다(지원 목록: random/even/texttiling/
  bayesseg/greedyseg/csm…). 학습된 체크포인트도 미배포(README TODO 에 `MTRoBERTa`/
  `MVRoBERTa` 가 `[ ]` 미구현).
- 따라서 **byte 단위 완전 재현은 불가능**합니다 — 학습 코드·고정 seed 가 공개된 적이
  없고, 논문도 "여러 번 돌려 평균" 이라 명시. 본 노트북은 *방법(method) 충실 재현*:
  아키텍처·입력 구성·하이퍼파라미터를 논문/repo 그대로 맞춥니다.
- 재현 대상 = 논문 **Table 3 의 `RoBERTa`** (plain supervised). `RobertaForTokenClassification`
  기반. repo `supervised.py` 의 `RobertaMultiTask`(=MT, da/role 출력 헤드 추가)·논문
  `MVRoBERTa`(=MV, role/da 입력 임베딩 추가)는 별도 변형 — 본 노트북은 깨끗한 기준선인
  plain RoBERTa 를 구현(확장 지점은 코드에 주석).
- **clone 은 `SuperDialseg` repo 하나뿐.** Hi-OnTop repo 는 clone 하지 않습니다.

### 논문 보고 수치 (Table 3, SuperDialseg 학습 · 재현 목표)

| 평가셋 | Pk↓ | WD↓ | F1↑ | Score↑ |
|---|---:|---:|---:|---:|
| SuperDialseg (in-domain) | 0.185 | 0.192 | 0.784 | 0.798 |
| TIAGE (zero-shot) | 0.401 | 0.443 | 0.373 | 0.482 |
| Dialseg711 (zero-shot) | 0.241 | 0.272 | 0.660 | 0.702 |

> 런타임 > 런타임 유형 변경 > **GPU** 로 설정 후 위에서부터 실행하세요.

## 1. GPU 확인

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임 유형을 GPU 로 변경하세요."

## 2. SuperDialseg repo clone + 의존성 설치

`SuperDialseg` repo 하나만 clone 합니다. repo 패키지(`pip install -e .`)는 구버전
의존성을 끌어와 Colab 과 충돌하므로 설치하지 않습니다 — 학습 코드는 이 노트북이
repo 코드대로 직접 구현하고, clone 은 원본 대조용입니다.

In [ ]:
import os
if not os.path.isdir("SuperDialseg"):
    !git clone --depth 1 https://github.com/Coldog2333/SuperDialseg
!pip -q install "gdown>=5.1" "transformers>=4.40,<4.50" "nltk>=3.8" scikit-learn
print("\n대조용 원본:",
      "SuperDialseg/src/super_dialseg/models/supervised.py,",
      "utils/data/data_collator.py")

## 3. SuperDialseg 데이터셋 다운로드 (Google Drive)

데이터셋은 repo 가 아니라 Google Drive 에 있습니다. `gdown` 으로 폴더째 받습니다.

In [ ]:
DRIVE_FOLDER = "https://drive.google.com/drive/folders/19YiHVfeI_M4HivrErIi9bghvUvsw9-Ws"
if not os.path.isdir("superdialseg_data"):
    !gdown --folder "{DRIVE_FOLDER}" -O ./superdialseg_data --remaining-ok
!find superdialseg_data -name 'segmentation_file_*.json' | sort

## 4. 데이터 위치 탐색

`segmentation_file_<split>.json` 을 찾아 `데이터셋 → split → 경로` 로 정리합니다
(부모 폴더 이름 = 데이터셋 이름; 보통 `superseg` / `tiage` / `dialseg711`).

In [ ]:
import glob
def discover_data(root="."):
    found = {}
    for p in glob.glob(os.path.join(root, "**", "segmentation_file_*.json"),
                       recursive=True):
        ds = os.path.basename(os.path.dirname(p))
        split = os.path.basename(p)[len("segmentation_file_"):-len(".json")]
        found.setdefault(ds, {})[split] = p
    return found

DATA = discover_data()
assert DATA, ("segmentation_file_*.json 미발견 — 3번 셀 다운로드 확인: " + DRIVE_FOLDER)
for ds in sorted(DATA):
    print(f"  {ds:12s}: {sorted(DATA[ds])}")

## 5. 재현 스펙 (논문 Appendix A + repo `data_collator.py`)

**입력 구성** — `DataCollatorForSupervisedDialogueSegmentation.__getitem_input__`
의 `roberta-base` 분기(`use_mask=False`)를 그대로 재현:

```
<s> u1_tok </s> </s> u2_tok </s> </s> ... uN_tok </s>
```
- 발화 1개는 `tokenizer.tokenize` 로 BPE 토큰화 후 **23개**(`max_utterance_len(25) − 2`)로 절단.
- 발화 뒤에 `</s> </s>` 2개를 붙이고, **첫 `</s>` 위치에서 분류**(`classification_mask=1`).
  논문: "two separate `</s>` … use the first `</s>` token to do classification."
- 마지막 `</s>` 1개는 제거(`input_tokens[:-1]`). 전체를 `max_seq_len=512` 로 절단/패딩.

**라벨** — token-classification. 첫 `</s>` 위치에 해당 발화의 `segmentation_label`,
나머지는 `-100`(ignore). 학습 시 윈도우 마지막 발화 라벨은 `-100`.

**슬라이딩 윈도우** — 발화 단위 `|T| = sliding_window = 20`, stride 1.
- 학습: 대화 길이 > 20 이면 임의 시작점에서 `sliding_window − 1 = 19` 발화 슬라이스
  (매 접근마다 다른 윈도우 → augmentation). 짧으면 전체 사용.
- 추론: stride-1 로 20-발화 윈도우 전부 생성. 한 발화가 여러 윈도우에 등장 →
  **logit 평균** 후 argmax (이 집계 방식은 논문·repo 에 미명시 → 표준 선택, 아래
  주석에 명시).

**모델** — `RobertaForTokenClassification`, `num_labels=2`.

**학습 HP** (논문 Appendix A): AdamW, **lr=1e-5, batch_size=8, weight_decay=1e-3**,
SuperDialseg **20 epochs** / TIAGE 40 epochs, early stopping(Score 가 전체 epoch 의
절반 동안 미개선 시 중단). LR 스케줄러·warmup 미언급 → 미사용.

**metric** — Pk / WindowDiff (sliding window = 평균 segment 길이의 절반),
binary F1, `Score = (2·F1 + (1−Pk) + (1−WD)) / 4`.

## 6. 학습 코드 (논문/repo 충실 재현, 인라인)

In [ ]:
import json, time, random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, RobertaForTokenClassification
from nltk.metrics import pk as _nltk_pk
from nltk.metrics import windowdiff as _nltk_wd
from sklearn.metrics import f1_score

IGNORE = -100


# ---- data ---------------------------------------------------------------
def load_dialogs(path):
    """SuperDialseg JSON -> [(utterances, segmentation_labels)]. yt[-1]=0 강제."""
    raw = json.loads(open(path, encoding="utf-8").read())
    arr = raw["dial_data"][list(raw["dial_data"])[0]]
    out = []
    for d in arr:
        utts = [t["utterance"] for t in d["turns"]]
        yt = [int(t.get("segmentation_label", 0)) for t in d["turns"]]
        if yt:
            yt[-1] = 0
        if len(utts) >= 2:
            out.append((utts, yt))
    return out


# ---- metric (논문 5.3: window = 평균 segment 길이의 절반) -----------------
def official_pk_wd(yt, yp):
    n_seg = sum(yt) + 1
    k = max(2, int(round(len(yt) / n_seg / 2)))
    ts, ps = "".join(map(str, yt)), "".join(map(str, yp))
    return float(_nltk_pk(ts, ps, k=k)), float(_nltk_wd(ts, ps, k=k))


def score_dialogs(dialogs, preds):
    pks, wds, g, p = [], [], [], []
    for (_, yt), yp in zip(dialogs, preds):
        pk, wd = official_pk_wd(yt, yp)
        pks.append(pk); wds.append(wd); g += yt; p += yp
    f1 = float(f1_score(g, p, zero_division=0))
    pk_m, wd_m = float(np.mean(pks)), float(np.mean(wds))
    return dict(pk=pk_m, wd=wd_m, f1=f1,
                score=(2 * f1 + (1 - pk_m) + (1 - wd_m)) / 4)


# ---- 입력 구성: data_collator.py __getitem_input__ roberta 분기 재현 -----
def encode_window(tok, utts, labels, mode, max_utt_len, max_seq_len):
    """<s> u1 </s></s> u2 </s></s> ... uN </s>  (마지막 </s> 제거).
    첫 </s> 위치(classification_mask=1)에 발화 라벨, 그 외 -100.
    반환: (input_ids, token_labels, cls_positions[utt_idx -> seq_idx])."""
    input_tokens = [tok.cls_token]      # '<s>'
    cmask = [0]
    for u in utts:
        ut = tok.tokenize(u)[:max_utt_len - 2]      # 25-2 = 23 (use_mask=False)
        input_tokens += ut + ["</s>", "</s>"]
        cmask += [0] * len(ut) + [1, 0]             # 첫 </s> 에서 분류
    input_tokens = input_tokens[:-1]                # 마지막 </s> 제거
    cmask = cmask[:-1]

    input_ids = tok.convert_tokens_to_ids(input_tokens)[:max_seq_len]
    cmask = cmask[:max_seq_len]

    lab = list(labels)
    if mode == "train" and lab:
        lab[-1] = IGNORE                            # 윈도우 마지막 발화 제외

    token_labels = [IGNORE] * len(input_ids)
    cls_pos = []                                    # utt i 의 분류 토큰 seq 위치
    i = 0
    for j, m in enumerate(cmask):
        if m:
            if i < len(lab):
                token_labels[j] = lab[i]
            cls_pos.append(j)
            i += 1
    return input_ids, token_labels, cls_pos


# ---- 학습 Dataset: __getitem__ 마다 임의 윈도우 (repo train 분기) --------
class TrainDS(Dataset):
    def __init__(self, dialogs, tok, cfg):
        self.dialogs, self.tok, self.cfg = dialogs, tok, cfg

    def __len__(self):
        return len(self.dialogs)

    def __getitem__(self, idx):
        utts, yt = self.dialogs[idx]
        W = self.cfg["sliding_window"]
        if len(utts) > W:
            s = random.randint(0, len(utts) - W)
            utts, yt = utts[s:s + W - 1], yt[s:s + W - 1]    # 19 발화
        ids, lab, _ = encode_window(self.tok, utts, yt, "train",
                                    self.cfg["max_utt_len"], self.cfg["max_seq_len"])
        return torch.tensor(ids), torch.tensor(lab)


def make_collate(pad_id):
    def collate(batch):
        ids, labs = zip(*batch)
        ids = pad_sequence(ids, batch_first=True, padding_value=pad_id)
        labs = pad_sequence(labs, batch_first=True, padding_value=IGNORE)
        attn = ids.ne(pad_id).long()
        return ids, attn, labs
    return collate


# ---- 추론: stride-1 윈도우 전부 → 발화별 logit 평균 → argmax -------------
@torch.no_grad()
def eval_set(model, tok, dialogs, cfg, device):
    model.eval()
    W, pad = cfg["sliding_window"], tok.pad_token_id
    preds = []
    for utts, yt in dialogs:
        n = len(utts)
        logit_sum = np.zeros((n, 2), dtype=np.float64)
        cnt = np.zeros(n, dtype=np.float64)
        win_ids, win_meta = [], []          # (window start, cls_pos)
        for ws in range(max(1, n - W + 1)):
            wu = utts[ws:ws + W]
            ids, _, cls_pos = encode_window(tok, wu, [0] * len(wu), "test",
                                            cfg["max_utt_len"], cfg["max_seq_len"])
            win_ids.append(torch.tensor(ids))
            win_meta.append((ws, cls_pos))
        # 한 대화의 윈도우들을 한 번에 forward
        batch = pad_sequence(win_ids, batch_first=True, padding_value=pad).to(device)
        attn = batch.ne(pad).long()
        logits = model(input_ids=batch, attention_mask=attn).logits.cpu().numpy()
        for w, (ws, cls_pos) in enumerate(win_meta):
            for u_local, j in enumerate(cls_pos):
                g = ws + u_local
                if g < n:
                    logit_sum[g] += logits[w, j]
                    cnt[g] += 1
        cnt[cnt == 0] = 1.0
        yp = list((logit_sum / cnt[:, None]).argmax(1))
        yp[-1] = 0                          # 마지막 발화는 경계 아님 (yt 규약)
        preds.append([int(x) for x in yp])
    return score_dialogs(dialogs, preds)


# 논문 Table 3 (SuperDialseg 학습) RoBERTa 보고 수치 — 재현 목표
PAPER_REF = {"superseg":   dict(pk=0.185, wd=0.192, f1=0.784, score=0.798),
             "tiage":      dict(pk=0.401, wd=0.443, f1=0.373, score=0.482),
             "dialseg711": dict(pk=0.241, wd=0.272, f1=0.660, score=0.702)}


def run(cfg):
    device = torch.device("cuda")
    random.seed(cfg["seed"]); np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"]); torch.cuda.manual_seed_all(cfg["seed"])

    tok = AutoTokenizer.from_pretrained(cfg["backbone"])
    tr = cfg["train_ds"]
    assert tr in DATA and "train" in DATA[tr], f"{tr}/train 없음: {DATA.get(tr)}"
    train_d = load_dialogs(DATA[tr]["train"])
    val_d = load_dialogs(DATA[tr]["validation"]) if "validation" in DATA[tr] else []
    test_sets = {ds: load_dialogs(DATA[ds]["test"])
                 for ds in DATA if "test" in DATA[ds]}
    if cfg["limit"]:
        train_d = train_d[:cfg["limit"]]
        val_d = val_d[:max(1, cfg["limit"] // 4)]
    print(f"[data] train {len(train_d)} / val {len(val_d)} dial (train_ds={tr})"
          f" | test: {sorted(test_sets)}")
    assert val_d, f"{tr}/validation 없음 — early stopping 에 필요"

    model = RobertaForTokenClassification.from_pretrained(
        cfg["backbone"], num_labels=2).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                              weight_decay=cfg["weight_decay"])
    loader = DataLoader(TrainDS(train_d, tok, cfg), batch_size=cfg["batch_size"],
                        shuffle=True, collate_fn=make_collate(tok.pad_token_id))

    out_dir = os.path.join("roberta_seg_out", cfg["name"])
    os.makedirs(out_dir, exist_ok=True)
    model_dir = os.path.join(out_dir, "model")
    patience = max(1, cfg["epochs"] // 2)       # 논문: 전체 epoch 의 절반
    best, since, history = -1.0, 0, []

    for epoch in range(1, cfg["epochs"] + 1):
        model.train(); t0 = time.perf_counter(); tot = 0.0
        for step, (ids, attn, labs) in enumerate(loader):
            out = model(input_ids=ids.to(device), attention_mask=attn.to(device),
                        labels=labs.to(device))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step(); optim.zero_grad()
            tot += out.loss.item()
        vr = eval_set(model, tok, val_d, cfg, device)
        history.append(dict(epoch=epoch, train_loss=tot / len(loader), val=vr))
        print(f"[epoch {epoch:>2}] loss={tot/len(loader):.4f} "
              f"val Score={vr['score']:.4f} (Pk={vr['pk']:.4f} F1={vr['f1']:.4f}) "
              f"{time.perf_counter()-t0:.0f}s")
        if vr["score"] > best:
            best, since = vr["score"], 0
            model.save_pretrained(model_dir); tok.save_pretrained(model_dir)
            print(f"   -> best 저장 (val Score={best:.4f})")
        else:
            since += 1
            if since >= patience:
                print(f"   early stopping (patience {patience} epoch 미개선)")
                break

    print("\n[final] best 체크포인트로 test 평가")
    best_model = RobertaForTokenClassification.from_pretrained(model_dir).to(device)
    results = {}
    for n, d in test_sets.items():
        r = eval_set(best_model, tok, d, cfg, device)
        results[n] = r
        ref = PAPER_REF.get(n, {})
        rs = f"  (논문 RoBERTa Score={ref.get('score')})" if ref else ""
        print(f"  {n:11s}: Pk={r['pk']:.4f} WD={r['wd']:.4f} F1={r['f1']:.4f} "
              f"Score={r['score']:.4f}{rs}")

    json.dump(dict(config=cfg, history=history, test=results,
                   paper_ref=PAPER_REF),
              open(os.path.join(out_dir, "results.json"), "w"), indent=2)
    print(f"\nDONE — 결과 {out_dir}/results.json · 모델 {model_dir}")
    return results

## 7. 학습 실행

논문 설정 그대로 `superseg` train 으로 학습 → 세 test split 평가.

- **빠른 동작 확인**: `LIMIT = 200`, `EPOCHS = 2` 로 파이프라인 검증(수 분).
- **본 재현**: `LIMIT = 0`(전체), `EPOCHS = 20`(논문 SuperDialseg 설정).
  A100 기준 1시간 이내, Colab T4/L4 는 더 걸릴 수 있습니다.

In [ ]:
LIMIT  = 200    # 0 = 전체 (논문 재현). 처음엔 200 으로 smoke test.
EPOCHS = 2      # 본 재현은 20 (SuperDialseg) / 40 (TIAGE).

cfg = dict(
    name="roberta_supervised",
    backbone="roberta-base",      # 논문 재현 대상
    train_ds="superseg",          # 학습 데이터셋 (train+validation split 필요)
    epochs=EPOCHS,
    batch_size=8,                 # 논문 Appendix A
    lr=1e-5,                      # 논문 Appendix A
    weight_decay=1e-3,            # 논문 Appendix A (L2-regularization)
    sliding_window=20,            # 논문 |T| = 20
    max_utt_len=25,               # 논문: 발화당 최대 25 토큰
    max_seq_len=512,              # call_tokenizer 기본값
    limit=LIMIT,
    seed=42,                      # repo main.py 기본 seed
)
results = run(cfg)

## 8. 결과 — 논문 수치 대조

`superseg` = in-domain, `tiage`/`dialseg711` = zero-shot transfer.
byte 단위 일치는 기대하지 않습니다(학습 코드·seed 미공개, 논문은 다회 평균) — 논문
RoBERTa 행과 **run variance 범위 내 근접**이면 충실 재현 성공으로 봅니다.

In [ ]:
import pandas as pd
rows = []
for n, r in results.items():
    ref = PAPER_REF.get(n, {})
    rows.append(dict(benchmark=n,
                     Pk=round(r["pk"], 4), WD=round(r["wd"], 4),
                     F1=round(r["f1"], 4), Score=round(r["score"], 4),
                     paper_Score=ref.get("score"),
                     gap=(round(r["score"] - ref["score"], 4) if ref else None)))
print(pd.DataFrame(rows).set_index("benchmark"))

## (선택) 학습한 모델을 Google Drive 에 저장

Colab 런타임이 끊기면 `roberta_seg_out/` 은 사라집니다. 보존하려면 주석을 푸세요.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r roberta_seg_out "/content/drive/MyDrive/roberta_seg_out"
# print("저장 완료: MyDrive/roberta_seg_out")